# Black-Scholes: put-call parity and price sensitivity to spot

Two checks on `pricing.black_scholes.black_scholes_price`:

1. **Put-call parity**: $C - P = S e^{-qT} - K e^{-rT}$ must hold numerically for any valid inputs.
2. **Price vs. spot**: plot call and put price as a function of the underlying price $S$, holding everything else fixed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pricing.black_scholes import black_scholes_price

# Fixed seed for reproducibility of any randomness introduced later in the notebook.
RNG_SEED = 42
np.random.seed(RNG_SEED)

## 1. Put-call parity check

In [ ]:
# Market parameters
S0 = 100.0   # spot price
K = 100.0    # strike
T = 1.0      # time to maturity (years)
r = 0.05     # risk-free rate
sigma = 0.20 # volatility
q = 0.02     # continuous dividend yield

call_price = black_scholes_price(S0, K, T, r, sigma, option_type="call", q=q)
put_price = black_scholes_price(S0, K, T, r, sigma, option_type="put", q=q)

lhs = call_price - put_price
rhs = S0 * np.exp(-q * T) - K * np.exp(-r * T)

print(f"Call price: {call_price:.6f}")
print(f"Put price:  {put_price:.6f}")
print(f"C - P (lhs):            {lhs:.6f}")
print(f"S*exp(-qT) - K*exp(-rT) (rhs): {rhs:.6f}")
print(f"Absolute difference:    {abs(lhs - rhs):.2e}")

assert np.isclose(lhs, rhs, atol=1e-10), "Put-call parity violated!"

## 2. Price vs. spot

In [ ]:
spot_grid = np.linspace(50, 150, 200)
call_prices = [black_scholes_price(S, K, T, r, sigma, option_type="call", q=q) for S in spot_grid]
put_prices = [black_scholes_price(S, K, T, r, sigma, option_type="put", q=q) for S in spot_grid]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(spot_grid, call_prices, label="Call")
ax.plot(spot_grid, put_prices, label="Put")
ax.axvline(K, color="gray", linestyle="--", linewidth=0.8, label="Strike K")
ax.set_xlabel("Spot price S")
ax.set_ylabel("Option price")
ax.set_title("Black-Scholes price vs. underlying spot")
ax.legend()
fig.tight_layout()
plt.show()